<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><ul class="toc-item"><li><span><a href="#Change-the-width-of-the-page" data-toc-modified-id="Change-the-width-of-the-page-0.1"><span class="toc-item-num">0.1&nbsp;&nbsp;</span>Change the width of the page</a></span></li><li><span><a href="#Import-packages" data-toc-modified-id="Import-packages-0.2"><span class="toc-item-num">0.2&nbsp;&nbsp;</span>Import packages</a></span></li></ul></li><li><span><a href="#Import-read-depths-from-control-samples" data-toc-modified-id="Import-read-depths-from-control-samples-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Import read depths from control samples</a></span></li><li><span><a href="#Import-BED-file" data-toc-modified-id="Import-BED-file-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>Import BED file</a></span></li><li><span><a href="#Import-sample-normalised-read-depths" data-toc-modified-id="Import-sample-normalised-read-depths-3"><span class="toc-item-num">3&nbsp;&nbsp;</span>Import sample-normalised read depths</a></span></li><li><span><a href="#Normalise-using-PON-(panel-of-normals)" data-toc-modified-id="Normalise-using-PON-(panel-of-normals)-4"><span class="toc-item-num">4&nbsp;&nbsp;</span>Normalise using PON (panel of normals)</a></span></li><li><span><a href="#Plot-normalised-depth-coverage" data-toc-modified-id="Plot-normalised-depth-coverage-5"><span class="toc-item-num">5&nbsp;&nbsp;</span>Plot normalised depth coverage</a></span></li></ul></div>

# Building the panel of normals (PON)

Builds the panel of normals used to normalise read depth before mosaic
chromosomal alteration (mCA) calling, from QC-passing control samples.

Coverage varies systematically between capture probes — some regions are always
captured more efficiently than others. A PON captures that baseline from control
samples so that a real copy-number change can be distinguished from a probe that
simply always reads low.

**Inputs** — per-sample normalised read depths written by the CNV / mCA panel:

```
<sample>_<UDI>/CNV_read_depths/<sample>_sample_normalised_read_depths.txt
```

one per control sample (final timepoints), plus the panel BED and
`chromosome_ideogram_hg19.txt` from `pipeline_tools/`. The manuscript used 36
control samples.

**Outputs** — into `CNV_panel_final_timepoint_read_depths/PON_normalised_read_depths/`:

| file | contents |
|---|---|
| `<sample>_PON_normalised_read_depths_and_LRR.txt` | per-sample PON-normalised depth and log-R ratio — the input to mCA calling |
| `PON_QC_Sample_CV_Values.csv` | per-sample coefficient of variation, for excluding noisy controls |
| `PON_QC_Distance_From_Threshold.csv`, `PON_QC_Visual_heatmp_CVs.pdf`, `QC_Population_Stats.csv` | QC summaries |

Samples that are themselves in the PON are normalised **leave-one-out**, so a
sample is never normalised against itself.

**Environment**: `conda activate tetris-seq-analysis`. Requires `pandas`,
`numpy`, `scipy`, `matplotlib`, `seaborn` and `scikit-learn`.

**Data availability**: the per-sample read depths are individual-level
participant data and are not distributed with this code. They are available
under controlled access from the European Genome-phenome Archive; see the main
[README](../README.md).


## Import packages

In [ ]:
# imported packages
import csv
import os
import shutil
from datetime import date

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [ ]:
%pip install seaborn

In [ ]:
# Lists of colors for plots
c0 = (0.76, 0.76, 0.76)
c1 = (1.00, 0.18, 0.33);
c2 = (1.00, 0.23, 0.19);
c3 = (1.00, 0.58, 0.00);
c4 = (1.00, 0.80, 0.00);
c5 = (0.30, 0.85, 0.39);
c6 = (0.35, 0.78, 0.98);
c7 = (0.20, 0.67, 0.86);
c8 = (0.00, 0.48, 1.00);
c9 = (0.35, 0.34, 0.84);
c10 = (0.00, 0.31, 0.57);
c11 = (0.12, 0.29, 0.69);
c12 = (0.17, 0.17, 0.42);
c13 = (1.00, 1.00, 1.00);
c14 = (0.77, 0.04, 0.00);

#define the colors from colorbrewer2
orange1 = '#feedde'
orange2 = '#fdbe85'
orange3 = '#fd8d3c'
orange4 = '#e6550d'
orange5 = '#a63603'
blue1 = '#eff3ff'
blue2 = '#bdd7e7'
blue3 = '#6baed6'
blue4 = '#3182bd'
blue5 = '#08519c'
green1 = '#edf8e9'
green2 = '#bae4b3'
green3 = '#74c476'
green4 = '#31a354'
green5 = '#006d2c'
grey1 = '#f7f7f7'
grey2 = '#cccccc'
grey3 = '#969696'
grey4 = '#636363'
grey5 = '#252525'
purple1 = '#f2f0f7'
purple2 = '#cbc9e2'
purple3 = '#9e9ac8'
purple4 = '#756bb1'
purple5 = '#54278f'
red1 = '#fee5d9'
red2 = '#fcae91'
red3 = '#fb6a4a'
red4 = '#de2d26'
red5 = '#a50f15'

# Import read depths from control samples

In [ ]:
final_timepoint_controls = ['CNTRL_169_s7'
'CNTRL_188_s7',
'CNTRL_181_s7',
'CNTRL_199_s7',
'CNTRL_160_s8',
'CNTRL_170_s6',
'CNTRL_204_s5',
'CNTRL_166_s4',
'CNTRL_177_s4',
'CNTRL_174_s3',
'CNTRL_186_s4',
'CNTRL_182_s4',
'CNTRL_200_s2',
'CNTRL_193_s2',
'CNTRL_161_s10',
'CNTRL_172_s7',
'CNTRL_194_s8',
'CNTRL_192_s7',
'CNTRL_191_s7',
'CNTRL_164_s6',
'CNTRL_165_s6',
'CNTRL_197_s6',
'CNTRL_202_s5',
'CNTRL_190_s4',
'CNTRL_180_s3',
'CNTRL_175_s3',
'CNTRL_173_s3',
'CNTRL_171_s2',
'CNTRL_187_s2',
'CNTRL_184_s5',
'CNTRL_167_s2',
'CNTRL_163_s6',
'CNTRL_002_s8',
'CNTRL_001_s10',
'CNTRL_004_s10',
'CNTRL_005_s9',
'CNTRL_179_s8',
'CNTRL_196_s8',
'CNTRL_003_s9',
'CNTRL_185_s4',
'CNTRL_162_s5',
'CNTRL_189_s4',
'CNTRL_168_s4',
'CNTRL_195_s3',
'CNTRL_198_s2'] #not including CNTRL_203_s5 as grossly abnormal BAF plots

## Create PON data file including all final timepoint controls except those with 1 or more chromosomes where the CV is >1.5 sd above the mean CV

In [ ]:
# ==========================================
# CONFIGURATION
# ==========================================
# Since the notebook is in the same folder as the SLX folders, we use '.'
base_path = "." 
libraries = ['SLX_19285', 'SLX_20125', 'SLX_20127']

pon_data_frames = []   # To store depth values
cv_records = []        # To store CVs for filtering
processed_count = 0

print("🔍 Starting PON generation...")

# ==========================================
# 1. LOAD DATA & CALCULATE STATS
# ==========================================
for lib in libraries:
    lib_path = os.path.join(base_path, lib)
    
    if not os.path.exists(lib_path):
        print(f"⚠️  Skipping missing library: {lib}")
        continue
    
    # Get all sample folders in this library
    sample_folders = [f for f in os.listdir(lib_path) if os.path.isdir(os.path.join(lib_path, f))]
    
    for sample_name in sample_folders:
        # We only want Controls for the PON
        if not sample_name.startswith("CNTRL"):
            continue

        if not sample_name in final_timepoint_controls:
            continue
            
        # Construct path: Library -> Sample -> CNV_read_depths
        sample_dir = os.path.join(lib_path, sample_name, "CNV_read_depths")
        
        if not os.path.exists(sample_dir):
            continue

        # Check for both naming conventions
        # Option A: sample_normalised_read_depths.txt
        # Option B: sample_CNV_normalised_read_depths.txt
        file_path = os.path.join(sample_dir, f"{sample_name}_sample_normalised_read_depths.txt")
        if not os.path.exists(file_path):
            file_path = os.path.join(sample_dir, f"{sample_name}_CNV_sample_normalised_read_depths.txt")
        
        if not os.path.exists(file_path):
            print(f"  ❌ File not found for {sample_name}") # Uncomment to debug
            continue
            
        try:
            # Read file (Header is on line 6 based on your screenshot, so header=5)
            df = pd.read_csv(file_path, sep='\t', skiprows=5)
            
            # Validation: Ensure columns exist
            if 'normalised_region_depth' not in df.columns:
                print(f"  ⚠️  Header mismatch in {sample_name}, skipping.")
                continue

            # --- STANDARDIZE CHROMOSOME NAMES ---
            # Ensure column is string type first
            df['chromosome'] = df['chromosome'].astype(str)
            
            # Apply lambda: if it doesn't start with 'chr', add it.
            df['chromosome'] = df['chromosome'].apply(lambda x: x if x.startswith('chr') else f'chr{x}')
            
            # --- A. Store Data for Matrix ---
            # Extract relevant columns and rename depth column to sample name
            sample_col = df[['chromosome', 'start', 'stop', 'band', 'normalised_region_depth']].copy()
            sample_col.rename(columns={'normalised_region_depth': sample_name}, inplace=True)
            sample_col.set_index(['chromosome', 'start', 'stop', 'band'], inplace=True)
            pon_data_frames.append(sample_col)
            
            # --- B. Calculate CV for Filtering ---
            # CV = Std Dev / Mean (per chromosome)
            stats = df.groupby('chromosome')['normalised_region_depth'].agg(['std', 'mean'])
            sample_cv = stats['std'] / stats['mean']
            sample_cv.name = sample_name
            cv_records.append(sample_cv)
            
            processed_count += 1
            
        except Exception as e:
            print(f"  ❌ Error reading {sample_name}: {e}")

print(f"✅ Successfully loaded {processed_count} control files.")

# ==========================================
# 2. APPLY FILTER (1.5 SD Rule)
# ==========================================

# Create DataFrame of CVs (Rows=Samples, Cols=Chromosomes)
cv_df = pd.concat(cv_records, axis=1).T 

# Calculate thresholds
mean_cv = cv_df.mean()
std_cv = cv_df.std()
thresholds = mean_cv + (1.5 * std_cv)

# Identify outliers
outlier_mask = cv_df > thresholds
failed_counts = outlier_mask.sum(axis=1)

# Filter: Keep samples with 0 failed chromosomes
valid_samples = failed_counts[failed_counts == 0].index.tolist()
excluded_samples = failed_counts[failed_counts >= 1].index.tolist()

print(f"\n📊 Filtering Results:")
print(f"   - Total Controls: {len(cv_df)}")
print(f"   - Passed: {len(valid_samples)}")
print(f"   - Excluded: {len(excluded_samples)}")

if excluded_samples:
    print(f"   - Excluded IDs: {excluded_samples}")

# ==========================================
# 3. SAVE FINAL MATRIX
# ==========================================
if valid_samples:
    full_pon = pd.concat(pon_data_frames, axis=1)
    final_pon = full_pon[valid_samples] # Keep only valid columns
    
    # Save to file
    final_pon.to_csv("CNV_panel_final_timepoint_read_depths/PON_normalised_read_depths/PON_master_matrix.tsv", sep='\t')
    print("\n💾 Success! Saved 'CNV_panel_final_timepoint_read_depths/PON_normalised_read_depths/PON_master_matrix.tsv'")
else:
    print("\n⚠️  No valid samples found!")

# ==========================================
# 4. QC & DEBUGGING OUTPUTS
# ==========================================

print("📊 Generating detailed QC reports...")

# --- A. Export Population Statistics ---
# Shows the Mean, SD, and Cutoff Threshold for each chromosome
pop_stats = pd.DataFrame({
    'Mean_CV': mean_cv,
    'Std_Dev': std_cv,
    'Threshold_1.5SD': thresholds
})
pop_stats.to_csv("QC_Population_Stats.csv")
print("   -> Saved 'QC_Population_Stats.csv' (Check this to see if thresholds are too loose)")

# --- B. Export Raw CVs ---
# Shows the calculated CV for every sample and chromosome
cv_df.to_csv("CNV_panel_final_timepoint_read_depths/PON_normalised_read_depths/PON_QC_Sample_CV_Values.csv")
print("   -> Saved 'CNV_panel_final_timepoint_read_depths/PON_normalised_read_depths/PON_QC_Sample_CV_Values.csv'")

# --- C. Export "Distance from Threshold" ---
# Positive values = FAILED (Above threshold)
# Negative values = PASSED (Below threshold)
# Values close to 0 (e.g. -0.01) mean it *barely* passed.
distance_df = cv_df - thresholds
distance_df.to_csv("CNV_panel_final_timepoint_read_depths/PON_normalised_read_depths/PON_QC_Distance_From_Threshold.csv")
print("   -> Saved 'CNV_panel_final_timepoint_read_depths/PON_normalised_read_depths/PON_QC_Distance_From_Threshold.csv' (Positive = Fail, Negative = Pass)")

# --- D. Visual Heatmap (Optional) ---
# This helps you visually spot if the 1.5 multiplier is too lenient
plt.figure(figsize=(15, 10))
# We plot the 'Distance' dataframe. Red = Bad (Above threshold), Blue = Good.
sns.heatmap(distance_df, cmap="vlag", center=0, cbar_kws={'label': 'Distance from Threshold'})
plt.title("Sample Performance relative to 1.5 SD Threshold\n(Red means excluded)")
plt.xlabel("Chromosome")
plt.ylabel("Sample")
plt.tight_layout()
plt.savefig('CNV_panel_final_timepoint_read_depths/PON_normalised_read_depths/PON_QC_Visual_heatmp_CVs.pdf')
plt.show()

## Checking for library specific variation in the PONs

In [ ]:
# ==========================================
# RECONSTRUCT SAMPLE MAP (Sample -> Library)
# ==========================================
sample_map = {}

print("🗺️  Mapping samples to libraries...")

for lib in libraries:
    lib_path = os.path.join(base_path, lib)
    
    if os.path.exists(lib_path):
        # Get all samples in this library folder
        samples_in_lib = os.listdir(lib_path)
        
        for s in samples_in_lib:
            # Only map samples that actually made it into our Final PON
            # (We don't care about excluded samples anymore)
            if s in final_pon.columns:
                sample_map[s] = lib

print(f"✅ Mapped {len(sample_map)} samples to their sequencing libraries.")
print(f"   Example: {list(sample_map.items())[0]}")


# ==========================================
# 5. GENERATE REGION STATISTICS (FOR QC/MASKING)
# ==========================================
print("📊 Generating Region Statistics (Mean, CV, Lane Breakdowns)...")

# 1. Overall Stats across all valid PON samples
region_stats = pd.DataFrame(index=final_pon.index)
region_stats['Global_Mean_Depth'] = final_pon.mean(axis=1)
region_stats['Global_Std_Dev'] = final_pon.std(axis=1)
region_stats['Global_CV'] = region_stats['Global_Std_Dev'] / region_stats['Global_Mean_Depth']

# 2. Breakdown by Library (Sequencing Lane)
# We use the sample names to identify the library (assuming you can map them)
# If you have a mapping dictionary `sample_map` from the earlier script:

for lib in libraries:
    # Identify columns (samples) belonging to this library
    # (This relies on the sample_map we created in the very first script block)
    # If you re-ran the script, ensure sample_map is populated.
    
    # robust way to find columns belonging to a library:
    lib_cols = [col for col in final_pon.columns if col in sample_map and sample_map[col] == lib]
    
    if lib_cols:
        lib_data = final_pon[lib_cols]
        region_stats[f'{lib}_Mean'] = lib_data.mean(axis=1)
        region_stats[f'{lib}_CV'] = lib_data.std(axis=1) / lib_data.mean(axis=1)

# 3. Save to file
region_stats.to_csv("CNV_panel_final_timepoint_read_depths/PON_normalised_read_depths/PON_Region_Stats.tsv", sep='\t')
print("✅ Saved 'CNV_panel_final_timepoint_read_depths/PON_normalised_read_depths/PON_Region_Stats.tsv'")
print("   Use this file to identify and mask noisy genomic regions (e.g. high CV regions).")

#### Plotting the data to look for library specific variation in PONs

In [ ]:
# ==========================================
# 1. PREPARE DATA FOR PCA (CLEAN UP)
# ==========================================
print(f"Original Matrix Shape: {final_pon.shape}")

# A. Drop Missing Values (NaNs)
# If a region is missing in even one sample, we drop it for the PCA visualization
clean_pon = final_pon.dropna()

# B. Drop Zero Variance Regions
# If a region has the exact same depth in all samples, standard deviation is 0.
# StandardScaler will try to divide by 0, creating NaNs. We must remove these.
clean_pon = clean_pon[clean_pon.std(axis=1) > 0]

print(f"Shape after cleaning: {clean_pon.shape}")

if clean_pon.shape[0] == 0:
    print("❌ ERROR: The cleaning process removed ALL rows.")
    print("   This means your samples might have totally disjoint regions (no overlap).")
    print("   Check if your chromosome names are standardized (chr1 vs 1)!")
else:
    # Transpose matrix so Samples are rows, Regions are columns
    data_for_pca = clean_pon.T

    # Map sample names to their library
    library_labels = [sample_map.get(s, 'Unknown') for s in data_for_pca.index]

    # Standardize the data
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(data_for_pca)
    
    # Now proceed to Step 2 (Run PCA)...
    print("✅ Data cleaned and scaled. Ready for PCA.")

# ==========================================
# 2. RUN PCA
# ==========================================
pca = PCA(n_components=2)
pca_result = pca.fit_transform(scaled_data)

# Create a DataFrame for plotting
pca_df = pd.DataFrame(data=pca_result, columns=['PC1', 'PC2'])
pca_df['Library'] = library_labels
pca_df['Sample'] = data_for_pca.index

# ==========================================
# 3. PLOT PCA (CHECK FOR BATCH EFFECTS)
# ==========================================
plt.figure(figsize=(8, 6))
sns.scatterplot(
    x='PC1', y='PC2',
    hue='Library',       # Color points by Library
    style='Library',     # Different shapes for libraries
    data=pca_df,
    s=100,               # Marker size
    alpha=0.8            # Transparency
)

plt.title('PCA of Control Samples (Check for Batch Effects)')
plt.xlabel(f'Principal Component 1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'Principal Component 2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.legend(title='Sequencing Library', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

# ==========================================
# 4. PLOT REGION CV HEATMAP (CHECK FOR NOISY REGIONS)
# ==========================================
# We will aggregate CVs by library to see if one library is noisier
region_cvs = pd.DataFrame()

for lib in set(library_labels):
    # Get samples for this library
    samples = [s for s, l in sample_map.items() if l == lib and s in final_pon.columns]
    if not samples: continue
    
    # Calculate CV for every region in this library
    lib_data = final_pon[samples]
    cv = lib_data.std(axis=1) / lib_data.mean(axis=1)
    region_cvs[lib] = cv

# Downsample for visualization if you have thousands of regions
# (Plotting 10,000 rows is slow and hard to read, so we take a sample or top variable regions)
if len(region_cvs) > 1000:
    # Sort by variance and take top 1000 most variable regions
    most_variable_regions = region_cvs.var(axis=1).sort_values(ascending=False).head(500).index
    plot_data = region_cvs.loc[most_variable_regions]
    title_suffix = "(Top 500 Variable Regions)"
else:
    plot_data = region_cvs
    title_suffix = "(All Regions)"

plt.figure(figsize=(8, 10))
sns.heatmap(plot_data, cmap="viridis", cbar_kws={'label': 'Coefficient of Variation (CV)'})
plt.title(f'Region Noise (CV) by Library {title_suffix}')
plt.ylabel("Genomic Regions")
plt.xlabel("Library")
plt.show()

PCA Plot Interpretation = No Batch Effects

If there were batch effects, you would see distinct islands—e.g., all the Blue dots clustered in one corner and all the Orange crosses in another.

The samples look thoroughly mixed. A blue dot is just as likely to be next to a green square as another blue dot.

This confirms that the biological variation (or random noise) between samples is greater than any technical variation introduced by the sequencing lane.

Conlusion = we can safely use the entire pool of controls to calculate your reference LRR, which gives you more statistical power.

Heatmap interpretation = consistent noise profile

We want to ensure one library isn't uniformly "noisier" (brighter/yellower) than the others.

The columns (libraries) look almost identical.

There are some distinct horizontal "stripes" that are lighter color across all three libraries. These represent genomic regions that are naturally difficult to sequence (likely centromeres, telomeres, or GC-rich regions).

Because these noisy regions are noisy in everyone, we can simply mask them out using the PON_Region_Stats.tsv file.

### Creating the PON-normalised read depth files per sample (with leave-one-out normalisation if sample also present in PON)

In [ ]:
# ==========================================
# 1. CONFIGURATION
# ==========================================
base_path = "."
# libraries = ['SLX_19285', 'SLX_20125', 'SLX_20127']
libraries = ['SLX_20550', 'SLX_20566', 'SLX_20567', 'SLX_20569']
pon_matrix_path = "CNV_panel_final_timepoint_read_depths/PON_normalised_read_depths/PON_master_matrix.tsv"
panel_bed_file = "TWIST_CNV_panel_TE-95031423_h19.bed"       
ideogram_file = "chromosome_ideogram_hg19.txt"  

output_folder_name = "PON_normalised_log2ratios_Feb2026"
today_str = date.today().strftime("%d/%m/%Y")

# ==========================================
# 2. DATA READERS & HELPERS
# ==========================================

def get_chrom_sizes_bands(ideogram_path):
    sizes = {}
    bands = {}
    if not os.path.exists(ideogram_path): return sizes, bands
    with open(ideogram_path, 'r') as f:
        reader = csv.reader(f, delimiter='\t')
        try:
            first = next(reader)
            if not first[0].startswith('#') and first[0] != 'chrom':
                c, s, e = first[0].replace('chr', ''), int(first[1]), int(first[2])
                sizes[c] = e
                bands.setdefault(c, []).append((s, e, first[3] if len(first)>3 else ''))
        except StopIteration: pass

        for row in reader:
            if not row: continue
            c, s, e = row[0].replace('chr', ''), int(row[1]), int(row[2])
            sizes[c] = e
            bands.setdefault(c, []).append((s, e, row[3] if len(row)>3 else ''))
    return sizes, bands

def get_panel_coverage(bed_path):
    coverage = {}
    if os.path.exists(bed_path):
        try:
            with open(bed_path, 'r') as f:
                reader = csv.reader(f, delimiter='\t')
                for row in reader:
                    if not row or row[0].startswith('#') or len(row) < 3: continue
                    try:
                        c = row[0].replace('chr', '')
                        s, e = int(row[1]), int(row[2])
                        coverage.setdefault(c, []).append((s, e))
                    except: continue
        except: pass
    return coverage

# ==========================================
# 3. PLOTTING FUNCTIONS
# ==========================================

def ideograms(ideogram_file, chromosome):
    
    color_lookup = {'gneg': (1., 1., 1.),
                    'gpos25': (.6, .6, .6),
                    'gpos50': (.4, .4, .4),
                    'gpos75': (.2, .2, .2),
                   'gpos100': (0., 0., 0.),
                      'acen': (.8, .4, .4),
                      'gvar': (.8, .8, .8),
                     'stalk': (.9, .9, .9)}
    
    ideogram = open(ideogram_file)
    ideogram.readline()
    xranges = []
    colors = []
    mid_points = []
    labels = []

    for line in ideogram:
        # print(line)
        chrom, start, stop, label, stain = line.strip().split('\t')
        start = int(start)
        stop = int(stop)
        width = stop - start
        mid_point = start + (width/2)
        if chrom == 'chr'+chromosome:
            xranges.append((start, width))
            colors.append(color_lookup[stain])
            mid_points.append(mid_point)
            labels.append(label)
        
    return xranges, [0, 0.9], colors, mid_points, labels

def plot_chromosome_ideogram(ideogram_file, chromosome, ax):

    xranges, yrange, colors, midpoints, labels = ideograms(ideogram_file, chromosome)

    ax.broken_barh(xranges, yrange, facecolors= colors, edgecolor = 'black')

    ax.set_xticks(midpoints)
    ax.set_xticklabels(labels, rotation = 90, fontsize = 9)
    ax.set_yticks([])
    ax.text(-0.013, 0.35, 'chr'+chromosome, transform=ax.transAxes, fontsize = 12, ha = 'right')
    ax.xaxis.set_tick_params(width=0.8, color = grey3, length = 6)

    ax.minorticks_off()

    ax.spines['left'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    
    return ax

def plot_chromosome(chrom, df_sample, sizes, bands, panel_cov, out_dir, sample_name, ideogram_file):
    c_key = chrom.replace('chr', '')
    if c_key not in sizes: return 

    subset = df_sample[df_sample.index.get_level_values('chromosome').astype(str).str.replace('chr', '') == c_key]
    if subset.empty: return

    starts = subset.index.get_level_values('start')
    stops = subset.index.get_level_values('stop')
    midpoints = (starts + stops) / 2
    lrr = subset['log2ratio']
    
    plt.close('all')
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 6), sharex=True, gridspec_kw={'height_ratios': [10, 1], 'hspace': 0.05})
    
    # --- 1. Panel Coverage Background (Light Blue Stripes) ---
    y_limit = max(abs(lrr.max()), abs(lrr.min()), 1.0) * 1.1
    y_min, y_max = -y_limit, y_limit
    
    if c_key in panel_cov:
        for (ps, pe) in panel_cov[c_key]:
            ax1.fill([ps, ps, pe, pe], [y_min, y_max, y_max, y_min], color='#deebf7', alpha=1.0, zorder=0)

    # --- 2. Scatter Plot (Blue Dots) ---
    ax1.scatter(midpoints, lrr, color='#4292c6', s=60, alpha=1.0, zorder=100)
    
    # --- 3. Red Mean Lines (Per Band) ---
    subset_reset = subset.reset_index()
    if 'band' in subset_reset.columns:
        band_groups = subset_reset.groupby('band')
        for band_name, grp in band_groups:
            band_mean = grp['log2ratio'].mean()
            b_start = grp['start'].min()
            b_stop = grp['stop'].max()
            ax1.plot([b_start, b_stop], [band_mean, band_mean], color='#ff2e54', lw=3, zorder=500)

    # --- 4. Highlights (P-value & CV) ---
    min_pval_mask = subset['p-value'] == subset['p-value'].min()
    if min_pval_mask.any():
        for start, stop in zip(starts[min_pval_mask], stops[min_pval_mask]):
            ax1.fill([start, start, stop, stop], [0, y_max, y_max, 0], color='#fdbe85', alpha=1.0, zorder=30)
    
    # High CV (Purple)
    cv_mask = subset['coefficient_of_variation_PON'] >= 0.2
    if cv_mask.any():
        for start, stop in zip(starts[cv_mask], stops[cv_mask]):
            ax1.fill([start, start, stop, stop], [y_min, 0, 0, y_min], color='#756bb1', alpha=1.0, zorder=40)

    # --- 5. Formatting ---
    ax1.set_ylim(y_min, y_max)
    ax1.set_xlim(0, sizes[c_key])
    ax1.set_ylabel('log2ratio normalised read depths', fontsize=12)
    ax1.set_title(f'Probe log2ratio read depths across chr{c_key} (from SSCS): {sample_name}', fontsize=14, fontweight='bold', y=1.12)
    
    ax1.axhline(0, color='grey', linestyle=':', lw=2)
    
    # Stats Box
    stats_text = (f"mean log2ratio = {lrr.mean():.1f}\n"
                  f"min log2ratio = {lrr.min():.1f}\n"
                  f"max log2ratio = {lrr.max():.1f}")
    ax1.text(0.02, 0.95, stats_text, transform=ax1.transAxes, fontsize=11, va='top', zorder=600)

    # Legend
    legend_elements = [
        Line2D([0], [0], marker='s', color='#deebf7', lw=0, markersize=10, label='regions covered by panel'),
        Line2D([0], [0], color='#ff2e54', lw=3, label='mean log2ratio across band'),
        Line2D([0], [0], marker='s', color='#fdbe85', lw=0, markersize=10, label='min p-value regions'),
        Line2D([0], [0], marker='s', color='#756bb1', lw=0, markersize=10, label='PONs CV >= 0.2')
    ]
    ax1.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, 1.12), ncol=4, fontsize=9, frameon=False)

    #Only show the required axis lines
    ax1.spines["bottom"].set_visible(False)
    ax2.spines["bottom"].set_visible(True)

    for ax in (ax1, ax2):
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_linewidth(1.5)
        ax.spines["left"].set_color(grey3)

    #x-axis ticks
    x1_major_ticks = []
    ax1.set_xticks(x1_major_ticks)
    ax1.tick_params(axis='y', which='major', labelsize=12)
    ax1.yaxis.set_tick_params(width=1, color = '#969696', length = 6)
    ax1.xaxis.set_tick_params(width=0, color = 'white', length = 0)

    # --- 6. Ideogram ---
    plot_chromosome_ideogram(ideogram_file, chrom, ax2)
    ax2.set_xlim(0, sizes[c_key])

    # Save
    filename = f"{sample_name}_CNV_logR_ratios_SSCS_chr{c_key}.pdf"
    plt.savefig(os.path.join(out_dir, filename), bbox_inches='tight')
    plt.close()

def plot_genome_wide(df_sample, sizes, panel_cov, out_dir, sample_name):
    """Plots all chromosomes stitched together."""
    ordered_chroms = [str(i) for i in range(1, 23)] + ['X'] # Original script omits Y in genome-wide usually
    
    # Calculate Offsets
    cum_sizes = {}
    total_len = 0
    ticks = []
    tick_labels = []
    
    for c in ordered_chroms:
        cum_sizes[c] = total_len
        mid = total_len + (sizes.get(c, 0) / 2)
        ticks.append(mid)
        tick_labels.append(c)
        total_len += sizes.get(c, 0)
        
    plt.close('all')
    fig, ax = plt.subplots(figsize=(20, 6))
    
    # Prepare Data with Offsets
    all_x = []
    all_y = []
    
    # Iterate for scatter
    for c in ordered_chroms:
        subset = df_sample[df_sample.index.get_level_values('chromosome').astype(str).str.replace('chr', '') == c]
        if subset.empty: continue
        
        offset = cum_sizes[c]
        midpoints = (subset.index.get_level_values('start') + subset.index.get_level_values('stop')) / 2
        all_x.extend(midpoints + offset)
        all_y.extend(subset['log2ratio'])
        
        # Plot Red Mean Lines
        subset_reset = subset.reset_index()
        if 'band' in subset_reset.columns:
            for band_name, grp in subset_reset.groupby('band'):
                mean_val = grp['log2ratio'].mean()
                x_start = grp['start'].min() + offset
                x_end = grp['stop'].max() + offset
                ax.plot([x_start, x_end], [mean_val, mean_val], color='#ff2e54', lw=3, zorder=500)

# --- 1. Panel Coverage Background (Light Blue Stripes) ---
    for c in ordered_chroms:
        subset = df_sample[df_sample.index.get_level_values('chromosome').astype(str).str.replace('chr', '') == c]
        offset = cum_sizes[c]
        y_limit = max(max(all_y), abs(min(all_y)), 1.0) * 1.1
        y_min, y_max = -y_limit, y_limit
        
        if c in panel_cov:
            for (ps, pe) in panel_cov[c]:
                ax.fill([ps+offset, ps+offset, pe+offset, pe+offset], [y_min, y_max, y_max, y_min], color='#deebf7', alpha=1.0, zorder=0)

    # Plot Scatter
    ax.scatter(all_x, all_y, color='#4292c6', s=25, alpha=1.0, zorder=100)
    
    # Formatting
    y_limit = max(max(all_y), abs(min(all_y)), 1.0) * 1.1
    ax.set_ylim(-y_limit, y_limit)
    ax.set_xlim(0, total_len)
    
    # Separator Lines
    for c in ordered_chroms:
        pos = cum_sizes[c]
        ax.axvline(x=pos, color='black', lw=1, zorder=1000)
    ax.axvline(x=total_len, color='black', lw=1, zorder=1000)
    
    ax.axhline(0, color='grey', linestyle=':', lw=2)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(1.5)
    ax.spines["left"].set_color(grey3)
    ax.spines["bottom"].set_linewidth(1.5)
    ax.spines["bottom"].set_color(grey3)
    ax.yaxis.set_tick_params(width=1, color = '#969696', length = 6)
    ax.xaxis.set_tick_params(width=1, color = '#969696', length = 6)
    
    ax.set_xticks(ticks)
    ax.set_xticklabels(tick_labels, fontsize=12)
    ax.set_xlabel('chromosome', fontsize=14)
    ax.set_ylabel('log2ratio normalised read depths', fontsize=14)
    ax.set_title(f'Probe log2ratio read depths (from SSCS): {sample_name}', fontsize=16, fontweight='bold', y=1.12)

    # Legend
    legend_elements = [
        Line2D([0], [0], marker='s', color='#deebf7', lw=0, markersize=10, label='regions covered by panel'),
        Line2D([0], [0], color='#ff2e54', lw=3, label='mean log2ratio across band')]
    ax.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, 1.12), ncol=4, fontsize=9, frameon=False)

    
    filename = f"{sample_name}_all_chromosomes.pdf"
    plt.savefig(os.path.join(out_dir, filename), bbox_inches='tight')
    plt.close()

# ==========================================
# 4. MAIN LOOP
# ==========================================

if not os.path.exists(pon_matrix_path):
    print(f"❌ Error: PON Matrix not found.")
    raise SystemExit

print("⏳ Loading PON and References...")
final_pon = pd.read_csv(pon_matrix_path, sep='\t', index_col=[0, 1, 2, 3])
chrom_sizes, chrom_bands = get_chrom_sizes_bands(ideogram_file)
panel_coverage = get_panel_coverage(panel_bed_file)

for lib in libraries:
    lib_path = os.path.join(base_path, lib)
    if not os.path.exists(lib_path): continue
    
    print(f"\n📂 Processing Library: {lib}")
    all_subfolders = [d for d in os.listdir(lib_path) if os.path.isdir(os.path.join(lib_path, d))]
    samples_to_process = [s for s in all_subfolders if s.startswith('CNTRL') or s.startswith('C92')]
    
    for i, sample in enumerate(samples_to_process):
        cnv_folder = os.path.join(lib_path, sample, "CNV_read_depths")
        if not os.path.exists(cnv_folder): continue
        
        # Robust File Search
        candidates = [f for f in os.listdir(cnv_folder) if sample in f and f.endswith("sample_normalised_read_depths.txt") and "PON" not in f]
        if not candidates:
             candidates = [f for f in os.listdir(cnv_folder) if f.endswith("sample_normalised_read_depths.txt") and "PON" not in f]
        if not candidates: continue

        candidates.sort(key=len)
        input_file = os.path.join(cnv_folder, candidates[0])

        out_root = os.path.join(lib_path, sample, output_folder_name)
        if os.path.exists(out_root): shutil.rmtree(out_root) # Clean start
        os.makedirs(out_root, exist_ok=True)
        pdf_dir = os.path.join(out_root, "PON_normalised_log2ratios_chromosome_PDFs")
        os.makedirs(pdf_dir, exist_ok=True)
        
        print(f"   [{i+1}/{len(samples_to_process)}] {sample} -> Generating Replica Plots...")

        try:
            # Read Data
            df_sample = pd.read_csv(input_file, sep='\t', skiprows=5)
            df_sample.columns = df_sample.columns.str.strip()
            
            # Standardize Chromosomes
            df_sample['chromosome'] = df_sample['chromosome'].astype(str).apply(lambda x: x if x.startswith('chr') else f'chr{x}')
            df_sample.set_index(['chromosome', 'start', 'stop', 'band'], inplace=True)
            
            # LOO Logic
            if sample in final_pon.columns: ref_set = final_pon.drop(columns=[sample])
            else: ref_set = final_pon
            
            common_index = df_sample.index.intersection(ref_set.index)
            sample_data = df_sample.loc[common_index]
            ref_data = ref_set.loc[common_index]

            pon_means = ref_data.mean(axis=1)
            pon_stds = ref_data.std(axis=1)
            pon_cvs = pon_stds / (pon_means + 1e-6) # Used for purple highlight
            
            obs_depth = sample_data['normalised_region_depth']
            norm_depth = obs_depth / pon_means
            log2ratios = np.log2(norm_depth) 
            
            # P-Values (Simplified Rank-based)
            n_controls = ref_data.shape[1]
            ranks = (ref_data.lt(obs_depth, axis=0)).sum(axis=1)
            dist = np.minimum(ranks, n_controls - ranks)
            p_values = (dist + 1) / (n_controls + 1) * 2
            
            # Compile Output
            output_df = sample_data.copy()
            output_df['log2ratio'] = log2ratios
            output_df['coefficient_of_variation_PON'] = pon_cvs
            output_df['p-value'] = p_values
            output_df['mean_normalised_depth_PON'] = pon_means
            output_df['sample_depth_normalised_by_PON'] = norm_depth

            # Save Text
            out_txt = os.path.join(out_root, f"{sample}_PON_normalised_read_depths_and_LRR.txt")
            with open(out_txt, 'w') as f:
                f.write(f"sample name :\t{sample}\n")
                f.write(f"date of analysis :\t{today_str}\n")
                f.write(f"produced from code:\tWatson_LOO_Replica_v3.0\n")
                f.write(f"total number of controls used in PON:\t{n_controls}\n\n")
                output_df.to_csv(f, sep='\t')
            
            # 1. Plot Each Chromosome (Detailed)
            chroms = [str(c) for c in range(1, 23)] + ['X', 'Y']
            for chrom in chroms:
                plot_chromosome(chrom, output_df, chrom_sizes, chrom_bands, panel_coverage, pdf_dir, sample, ideogram_file)
                
            # 2. Plot Genome Wide
            plot_genome_wide(output_df, chrom_sizes, panel_coverage, pdf_dir, sample)
                
        except Exception as e:
            print(f"   ❌ Error processing {sample}: {e}")
            continue

print("\n✅ Analysis Complete. Check sample folders.")

filtered out probes where min read depth was <0.2 median or maximum >3 median